In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from catboost import CatBoostRegressor, Pool
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/megaolimpiade-2026/sample_submission.csv
/kaggle/input/megaolimpiade-2026/train.csv
/kaggle/input/megaolimpiade-2026/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/megaolimpiade-2026/train.csv')
test = pd.read_csv('/kaggle/input/megaolimpiade-2026/test.csv')

## Предобработка данных

In [3]:
train.head(5)

,id,price_rub,mark,model,generation,configuration,complectation,body_type,color,displacement,...,owners_count,condition,custom,pts,seller_type,region,city,address,options,description
0,16799418,2200000,Kia,Sportage,IV Рестайлинг,Внедорожник 5 дв.,Prestige Black Edition,пятидверный внедорожник,чёрный,2.4,...,1,среднее,растаможен,оригинал,частник,Вологодская область,Череповец,"Россия, Вологодская область,","[""airbag-curtain"",""park-assist-r"",""leather"",""b...","Автомобиль в отличном состоянии, вложений ника..."
1,14261299,300000,Ford,Maverick,III,Внедорожник 5 дв.,NaN,пятидверный внедорожник,серебристый,3.0,...,4,среднее,растаможен,дубликат,частник,Краснодарский край,Сочи,Теневой переулок,NaN,"Есть жизненные повреждения, автомобиль рабочий..."
2,10850488,2600000,Volkswagen,Caravelle,T6,Минивэн,Trendline,минивэн,чёрный,2.0,...,2,среднее,растаможен,оригинал,частник,Оренбургская область,Орск,NaN,"[""isofix"",""front-seats-heat"",""hcc"",""heated-was...","Автомобиль в отличном состоянии, все то пройде..."
3,16407528,549000,Opel,Astra,H Рестайлинг,Седан,Essentia,седан,чёрный,1.6,...,4,среднее,растаможен,дубликат,частник,Самарская область,Самара,NaN,"[""abs"",""airbag-driver"",""airbag-passenger"",""air...",Продаю личный автомобиль в хорошем состоянии. ...
4,12695854,700000,Hyundai,Santa Fe,I,Внедорожник 5 дв.,NaN,пятидверный внедорожник,красный,2.7,...,3,среднее,растаможен,оригинал,частник,Краснодарский край,Краснодар,Адлерский район,NaN,"Пороги переваренные, грм новый, масла в двигат..."


В датасете есть достаточно много категриальных признаков, поэтому в качестве модели для предсказания будет исопользован регрессор `CatBoost`, так как надо предсказывать цену, которая является непрерывной величиной (поэтому регрессор).

Кроме того категориальные признаки в `CatBoost` можно отдельно обрабатывать. Поэтому выделим их в отдельный список `cat_features`.

Также есть такие признаки как адрес, список с доп опциями и описание объявления. Сделаем их отдельную обработку, а адрес исключим. Вместо адреса можно использовать фичу "наличие / отсутствие адреса".

In [4]:
cat_features = ['mark', 'model', 'generation', 'configuration', 'complectation',
               'body_type', 'color', 'drive_type', 'engine_type', 'transmission', 
               'wheel', 'condition', 'custom', 'pts', 'seller_type', 'region', 'city']
ignored_features = ['address', 'options', 'description']

In [5]:
def feature_engine(df_):
    df = df_.copy()
    mode_dict = {}
    for cat in tqdm(cat_features):
        mode_dict[cat] = df[cat].mode()
    
    for cat in tqdm(cat_features):
        df[cat] = df[cat].apply(lambda x: df[cat] if type(x) == float else x).astype(str)

    # обрабатываем опции -- вместо них будет их количество
    df['cnt_options'] = df.options.apply(lambda x: len(x) if type(x) != float else 0)
    # обрабатываем описание -- вместо описания берем его длину
    df['length_description'] = df.description.apply(lambda x: len(x) if type(x) != float else 0)
    # а также берем количество уникальных слов в описании как признак
    df['uniq_description'] = df.description.apply(lambda x: len(list(set(x.split(' ')))) if type(x) != float else 0)
    # делаем фичу -- наличие или отсутствие адреса
    df['has_adress'] = df.address.apply(lambda x: 1 if type(x) != float else 0)

    return df

In [6]:
X, y = feature_engine(train).drop(['price_rub', 'id'] + ignored_features, axis=1), train.price_rub

100%|██████████| 17/17 [00:34<00:00,  2.01s/it]


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [8]:
cbr = CatBoostRegressor(
    cat_features=cat_features,
    loss_function='MAE',
    one_hot_max_size=10
)

In [9]:
cbr.fit(X_train, y_train, cat_features=cat_features, verbose=False)

## feature_importance

In [10]:
get_feature_importance_list = cbr.get_feature_importance(Pool(X_train, cat_features=cat_features))

In [11]:
X.columns.to_numpy()[get_feature_importance_list.argsort()]

array(['custom', 'wheel', 'condition', 'has_adress', 'color',
       'length_description', 'uniq_description', 'city', 'complectation',
       'pts', 'seller_type', 'region', 'transmission', 'drive_type',
       'owners_count', 'configuration', 'body_type', 'cnt_options',
       'model', 'engine_type', 'generation', 'mark', 'km_age',
       'displacement', 'year', 'horse_power'], dtype=object)

In [12]:
get_feature_importance_list[get_feature_importance_list.argsort()]

array([ 0.        ,  0.04756717,  0.08446743,  0.10230471,  0.17463245,
        0.27815723,  0.3123522 ,  0.34142389,  0.43030819,  0.71716264,
        0.80375084,  0.83014477,  0.89250168,  1.23849115,  1.39624976,
        1.46186058,  1.63639636,  2.1227862 ,  2.45777858,  2.87883618,
        4.48085591,  6.2842351 , 14.91090206, 17.22692531, 17.78926954,
       21.10064007])

Можно попробовать удалить custom и wheel, чтобы посмотреть, станут ли предсказания лучше

In [13]:
y_pred = cbr.predict(X_test, verbose=False)

In [14]:
mean_absolute_error(y_test.to_list(), y_pred), r2_score(y_test.to_list(), y_pred)

(276541.5357221078, 0.8254654360019797)

## Удаляем малозначимые фичи

In [15]:
X, y = feature_engine(train).drop(['price_rub', 'id', 'custom', 'wheel', 'condition', 'has_adress'] + ignored_features, axis=1), train.price_rub

100%|██████████| 17/17 [00:33<00:00,  1.98s/it]


In [16]:
cat_features = ['mark', 'model', 'generation', 'configuration', 'complectation',
               'body_type', 'color', 'drive_type', 'engine_type', 'transmission', 
                'pts', 'seller_type', 'region', 'city']

In [17]:
cbr = CatBoostRegressor(
    cat_features=cat_features,
    loss_function='MAE',
    one_hot_max_size=10,
    l2_leaf_reg=7,
    learning_rate=0.1,
    max_depth=5
)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [19]:
# grid = {'learning_rate': [0.02, 0.2],
#         'depth': [4, 6]}

# grid_search_result = cbr.grid_search(grid,
#                                        X=X,
#                                        y=y,
#                                        verbose=False)

In [20]:
cbr.fit(X_train, y_train, cat_features=cat_features, verbose=False)

In [21]:
y_pred = cbr.predict(X_test, verbose=False)

In [22]:
mean_absolute_error(y_test.to_list(), y_pred), r2_score(y_test.to_list(), y_pred)

(270817.32956929767, 0.8493135394980456)

## Предсказание

In [23]:
X_true_test = feature_engine(test).drop(['id', 'custom', 'wheel', 'condition', 'has_adress'] + ignored_features, axis=1)

100%|██████████| 14/14 [00:08<00:00,  1.67it/s]


In [24]:
y_true_pred = cbr.predict(X_true_test, verbose=False)

In [25]:
pd.read_csv('/kaggle/input/megaolimpiade-2026/sample_submission.csv').head()

,id,price_rub
0,16531113,1.131430e+06
1,16501819,1.644392e+06
2,10910893,1.321285e+06
3,12258595,1.675514e+06
4,11086668,3.822850e+05


In [26]:
result = test[['id']]
result['price_rub'] = y_true_pred

/tmp/ipykernel_17/3780773981.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result['price_rub'] = y_true_pred


In [27]:
result.to_csv('sample_submission.csv', index=False)